# 05 — Gate AB-0 (ii): G0 checks, the S4 floor ladder, and `scenarios_ab_v1.json` (Python)

Runs AFTER `04_ab0_arms`. Kernel `y2y-geo`. **Zero solves.** Mirrors the parent's
`03_gate0_validation` (G0 checks) + `05_s0_construction` (carbon split diagnostic → S0–S5
derivation) with the Gate AB-0a rulings applied (methods_log M4):

- **D-AB9**: both AOH members stay in the biodiversity block (mammals' weight inert, disclosed).
- **D2**: S0–S3 keep m_soc t = 0.322 although pre-satisfied.
- **D3 + pre-registered ladder (spec §6, v0.4.3)**: S4's m_soc target = the first of θ ∈ {2, 1.5, 1.2}
  (archive lookup → 0.772 / 0.848 / 0.877) that exceeds the **measured** level-A floor
  (a0_control's m_soc capture) by ≥ 0.01. The zero-solve proxy (0.765) was a lower bound; the
  solve gives the real floor. The AB-1 S4 pilot certifies binding; if it does not bind, step up
  the ladder (pre-registered fallback, no new decision).
- **D-AB5 v2**: writes `spec/ab_budget_levels_v1.json` (levels A and B).

The G0 exit criteria are re-read for a pre-satisfied protocol target: a1's m_soc is expected
ABOVE its target (that is the E10 finding, not a failure); the kink test moves to a5.

In [9]:
# ---- bootstrap ------------------------------------------------------------------------------------
import importlib, json, pathlib, sys
from datetime import datetime, timezone
import numpy as np
import pandas as pd

_cands = [p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "config.py").exists()]
assert _cands, "config.py not found above the notebook"
ROOT = _cands[0]
sys.path.insert(0, str(ROOT))
import config, leverage_core as lc, ensemble_core as ec
for _m in (config, lc, ec):
    importlib.reload(_m)

HERE = ROOT / "analyses" / "alberta_prioritization"
SPEC, AUDIT_OBJ = HERE / "spec", HERE / "audit" / "audit_objects_ab"
GATE0 = HERE / "runs" / "ab_l" / "gate0"
AB = config.AB_HANDOFF_DIR
EXTENT = json.loads((SPEC / "ab_extent_v1.json").read_text())
CONSTS = json.loads((AUDIT_OBJ / "audit_constants.json").read_text())
T2 = pd.read_csv(AUDIT_OBJ / "feature_characterization.csv").set_index("feature")
SUP = pd.read_csv(AUDIT_OBJ / "supplementary_columns.csv").set_index("feature")
Z = np.load(AUDIT_OBJ / "feature_audit.npz")
assert CONSTS["constants"] == config.AUDIT, "config.AUDIT drifted from the frozen AB audit -- STOP"
assert CONSTS["n_pu"] == EXTENT["n_pu"]

ARMS = ["a0_control", "a1_protocol", "a4_pullcheck", "a5_floor"]
RUNS = {a: GATE0 / a for a in ARMS if (GATE0 / a / "run_summary.json").exists()}
missing = [a for a in ARMS if a not in RUNS]
assert not missing, f"arms not solved yet (run 04_ab0_arms): {missing}"
CAPTURE = {a: pd.read_csv(d / "portfolio_representation.csv").set_index("feature")["relative_held"] for a, d in RUNS.items()}
SUMMARY = {a: json.loads((d / "run_summary.json").read_text()) for a, d in RUNS.items()}
for a, s in SUMMARY.items():
    assert s["n_planning_units"] == EXTENT["n_pu"] and s["n_locked_in"] == EXTENT["n_locked"], a
    assert abs(s["budget_cells"] - EXTENT["budget_cells"]) <= 1, f"{a}: budget != level A"
print("arms:", ", ".join(f"{a} ({SUMMARY[a]['solve_seconds']:.0f}s)" for a in RUNS))
print(f"level A: PU {EXTENT['n_pu']:,} | locked {EXTENT['n_locked']:,} | budget {EXTENT['budget_cells']:,}")

# the typed arm targets must match the frozen archive (04 types them; this is the guard)
def theta_target(feature, th):
    ratio, cap = Z[f"{feature}__dens_ratio"], Z[f"{feature}__captured"]
    ab = ratio >= th
    return float(cap[ab][-1]) if ab.any() else 0.0
t_a1 = float(SUMMARY["a1_protocol"]["params"]["targets"]["irrecoverable_carbon_m_soc"])
t_a5 = float(SUMMARY["a5_floor"]["params"]["targets"]["irrecoverable_carbon_m_soc"])
assert abs(t_a1 - float(T2.loc["irrecoverable_carbon_m_soc", "target"])) < 1e-6, "a1 target != frozen T2"
assert abs(t_a5 - round(theta_target("irrecoverable_carbon_m_soc", 2.0), 3)) < 1e-6, "a5 target != theta-2x archive lookup"
t_a4 = float(SUMMARY["a4_pullcheck"]["params"]["targets"]["transboundary_connectivity"])
assert t_a4 > float(T2.loc["transboundary_connectivity", "cap_max_eff"]), "a4 target is NOT unreachable at the effective budget"
print(f"arm targets verified against the archive: a1 {t_a1} | a5 {t_a5} (theta 2x) | a4 {t_a4} > cap_max@eff {T2.loc['transboundary_connectivity','cap_max_eff']}")

arms: a0_control (1s), a1_protocol (1s), a4_pullcheck (1s), a5_floor (1s)
level A: PU 85,133 | locked 27,972 | budget 38,055
arm targets verified against the archive: a1 0.322 | a5 0.772 (theta 2x) | a4 0.8 > cap_max@eff 0.753


## G0 — captures vs targets, the measured floor, and the D3 ladder

In [10]:
# ---- G0.1 capture vs target per arm (the pre-satisfaction reading) ------------------------------
TOL = 0.005
POOLS = ["irrecoverable_carbon_m_soc", "irrecoverable_carbon_biomass"]
rows = []
for a in RUNS:
    tg = SUMMARY[a]["params"].get("targets") or {}
    for f in POOLS:
        t, c = float(tg.get(f, 1.0)), float(CAPTURE[a][f])
        v = ("free (no target)" if t >= 0.999 else
             "AT target (kink)" if abs(c - t) <= TOL else
             f"ABOVE target (+{c-t:.3f}) -- non-binding" if c > t else
             "BELOW target <-- INVESTIGATE")
        rows.append(dict(arm=a, feature=f.replace("irrecoverable_carbon_", ""), target=round(t, 3),
                         captured=round(c, 4), verdict=v, solve_s=round(SUMMARY[a]["solve_seconds"], 1)))
G1 = pd.DataFrame(rows)
print(G1.to_string(index=False))
assert not G1.verdict.str.contains("INVESTIGATE").any(), "a targeted pool is BELOW its target -- stopping point not reached; STOP"

# ---- G0.2 the measured level-A floor vs the zero-solve proxy --------------------------------------
# CORRECTION (M5.7, first run 2026-09-03): the E10 floor is co-capture + lock-in = capture when the
# carbon term exerts NO pull. a1's target (0.322) is pre-satisfied, so above it the term is zero and
# a1's realized m_soc capture IS that floor. a0 (equal weights) measures capture WITH carbon's own
# pull -- the wrong reference (0.867 here), which is why the first run's ladder assert fired.
a0_ms = float(CAPTURE["a0_control"]["irrecoverable_carbon_m_soc"])
a1_ms = float(CAPTURE["a1_protocol"]["irrecoverable_carbon_m_soc"])
BANKED = float(SUP.loc["irrecoverable_carbon_m_soc", "banked_capture"])
PROXY = float(SUP.loc["irrecoverable_carbon_m_soc", "window_floor_proxy"])
assert a1_ms > 0.322 + TOL, "a1's protocol target BINDS -- not pre-satisfied; the floor logic below does not apply, STOP"
FLOOR_A = a1_ms
print(f"\nm_soc: banked {BANKED:.3f} | zero-solve proxy floor {PROXY:.3f} | MEASURED co-capture floor (a1, zero carbon pull) {FLOOR_A:.4f}"
      f" | a0 equal-weight capture {a0_ms:.4f} (carbon WITH pull -- not the floor)")
print(f"a1 (t=0.322): capture {a1_ms:.4f} -> PRE-SATISFIED, target inert (parent E10 in the wild)")

# ---- G0.3 the pre-registered D3 ladder --------------------------------------------------------------
LADDER = [(2.0, round(theta_target("irrecoverable_carbon_m_soc", 2.0), 3)),
          (1.5, round(theta_target("irrecoverable_carbon_m_soc", 1.5), 3)),
          (1.2, round(theta_target("irrecoverable_carbon_m_soc", 1.2), 3))]
MARGIN = 0.01
T_S4 = next((t for th, t in LADDER if t >= FLOOR_A + MARGIN), None)
TH_S4 = next((th for th, t in LADDER if t >= FLOOR_A + MARGIN), None)
print("\nladder (theta -> target):", " | ".join(f"{th}x -> {t:.3f}" for th, t in LADDER))
assert T_S4 is not None, "no ladder rung clears the measured floor + margin -- back to the chat"
a5_ms = float(CAPTURE["a5_floor"]["irrecoverable_carbon_m_soc"])
print(f"a5 (t={t_a5}): capture {a5_ms:.4f} -> {'AT KINK -- binds' if abs(a5_ms - t_a5) <= TOL else 'above target -- does NOT bind at equal weights'}")
print(f"S4 m_soc target = {T_S4:.3f} (theta {TH_S4}x): first rung >= measured floor {FLOOR_A:.3f} + {MARGIN}")
if TH_S4 != 2.0:
    print("   NOTE: theta 2x did not clear the measured floor; ladder stepped up (pre-registered, no new decision)")

         arm feature  target  captured                              verdict  solve_s
  a0_control   m_soc   1.000    0.8671                     free (no target)      1.3
  a0_control biomass   1.000    0.4128                     free (no target)      1.3
 a1_protocol   m_soc   0.322    0.7438 ABOVE target (+0.422) -- non-binding      1.1
 a1_protocol biomass   1.000    0.4618                     free (no target)      1.1
a4_pullcheck   m_soc   1.000    0.8671                     free (no target)      1.1
a4_pullcheck biomass   1.000    0.4128                     free (no target)      1.1
    a5_floor   m_soc   0.772    0.7720                     AT target (kink)      1.5
    a5_floor biomass   1.000    0.4517                     free (no target)      1.5

m_soc: banked 0.714 | zero-solve proxy floor 0.765 | MEASURED co-capture floor (a1, zero carbon pull) 0.7438 | a0 equal-weight capture 0.8671 (carbon WITH pull -- not the floor)
a1 (t=0.322): capture 0.7438 -> PRE-SATISFIED, target in

In [11]:
# ---- G0.4 a4 pull-invariance (binary: objective-equivalence, parent convention) + reallocation ------
cont = lc.continuous_features()
efg = [f for f in CAPTURE["a0_control"].index if f not in cont and f != "irrecoverable_carbon_sl_soc"]
obj = lambda h: sum(1 - h[f] for f in cont) + sum((1 - h[f]) / len(efg) for f in efg)
o0, o4 = obj(CAPTURE["a0_control"]), obj(CAPTURE["a4_pullcheck"])
rel = abs(o4 - o0) / o0
g = float(SUMMARY["a4_pullcheck"]["params"].get("opt_gap", 1e-4))
S0sel = np.nan_to_num(ec._alloc(RUNS["a0_control"] / "portfolio.tif")) > 0.5
S4sel = np.nan_to_num(ec._alloc(RUNS["a4_pullcheck"] / "portfolio.tif")) > 0.5
j4 = (S4sel & S0sel).sum() / max((S4sel | S0sel).sum(), 1)
print(f"G-uniform (a4 vs a0): common objective {o0:.5f} vs {o4:.5f} (rel {rel:.3%}) | band {2.5*g:.2%} | "
      f"Jaccard {j4:.4f} | differing cells {int((S4sel != S0sel).sum()):,}")
print("   " + ("PASS -- objective-equivalent (differing cells = near-tie degeneracy datum)" if rel <= 2.5 * g
               else "INVESTIGATE -- exceeds the certificate band"))

tbl = pd.DataFrame({a: CAPTURE[a] for a in ("a0_control", "a1_protocol", "a5_floor")})
out = tbl.loc[cont].copy()
out.loc["EFG mean"] = tbl.loc[efg].mean()
for a in ("a1_protocol", "a5_floor"):
    out[f"d {a}"] = out[a] - out["a0_control"]
print("\ncaptures at level A and reallocation vs control:")
print(out.round(3).to_string())

G-uniform (a4 vs a0): common objective 4.20025 vs 4.20025 (rel 0.000%) | band 0.03% | Jaccard 1.0000 | differing cells 0
   PASS -- objective-equivalent (differing cells = near-tie degeneracy datum)

captures at level A and reallocation vs control:
                              a0_control  a1_protocol  a5_floor  d a1_protocol  d a5_floor
feature                                                                                   
human_modification                 0.472        0.472     0.472         -0.000      -0.000
transboundary_connectivity         0.527        0.547     0.547          0.020       0.020
climate_corridors                  0.477        0.477     0.477         -0.001      -0.001
climate_type_macrorefugia          0.565        0.556     0.558         -0.010      -0.008
irrecoverable_carbon_biomass       0.413        0.462     0.452          0.049       0.039
irrecoverable_carbon_m_soc         0.867        0.744     0.772         -0.123      -0.095
aoh_richness_mammals   

## §A — carbon within-block split: the biomass θ-tail diagnostic (parent Gate-1 rule, AB reading)

Parent rule (frozen): a1's mass-weighted capture of the biomass θ-tail ≥ 0.90 → mass-proportional
split; 0.50–0.90 → equal; < 0.50 → STOP. **AB caveat, disclosed:** the biomass θ-tail here is
~0.05% of the extent (θ-target 0.003), so the rule is near-vacuous — reported and applied as
written, the split it selects is stated with the caveat.

In [12]:
# ---- theta-tails on the AB PU + a1 tail capture --------------------------------------------------------
pu = lc.pu_mask(AB)
assert int(pu.sum()) == CONSTS["n_pu"]
theta = config.AUDIT["theta"]
bio = lc._read(AB / "irrecoverable_carbon_biomass.tif")
soc = lc._read(AB / "irrecoverable_carbon_m_soc.tif")
tails = {}
for name, arr in (("irrecoverable_carbon_biomass", bio), ("irrecoverable_carbon_m_soc", soc)):
    cut = theta * float(np.nanmean(arr[pu]))
    tails[name] = pu & (np.nan_to_num(arr, nan=-1.0) >= cut)
    print(f"{name}: cutoff {cut:.1f} t/ha | tail {int(tails[name].sum()):,} cells = {100*tails[name].sum()/pu.sum():.3f}% of PU "
          f"(frozen T2 {100*float(T2.loc[name,'theta_area']):.2f}%)")
bio_tail, soc_tail = tails["irrecoverable_carbon_biomass"], tails["irrecoverable_carbon_m_soc"]

sel = lambda a: np.nan_to_num(ec._alloc(RUNS[a] / "portfolio.tif")) > 0.5
mass_tot = float(np.nansum(bio[bio_tail]))
rows = []
for arm in ("a0_control", "a1_protocol"):
    s = sel(arm); hit = bio_tail & s
    rows.append(dict(arm=arm, tail_cells=f"{int(hit.sum()):,}/{int(bio_tail.sum()):,}",
                     capture_mass=float(np.nansum(bio[hit])) / mass_tot,
                     soc_claim_co_capture=float(np.nansum(bio[hit & soc_tail])) / mass_tot,
                     independent=float(np.nansum(bio[hit & ~soc_tail])) / mass_tot))
D = pd.DataFrame(rows).set_index("arm"); print(D.round(4).to_string())
MASS_SOC, MASS_BIO = float(np.nansum(soc[pu])), float(np.nansum(bio[pu]))
FRAC_SOC_MASS = MASS_SOC / (MASS_SOC + MASS_BIO)
print(f"AB carbon mass split: SOC {100*FRAC_SOC_MASS:.1f}% / biomass {100*(1-FRAC_SOC_MASS):.1f}% (parent 74.2/25.8)")
CAP = float(D.loc["a1_protocol", "capture_mass"])
SPLIT = "mass" if CAP >= 0.90 else "equal" if CAP >= 0.50 else "STOP"
print(f"a1 tail capture (mass) = {CAP:.3f} -> split rule: {SPLIT}  [near-vacuous at AB tail size; disclosed]")
assert SPLIT != "STOP", "tail escaping -- back to the chat"
frac_soc = FRAC_SOC_MASS if SPLIT == "mass" else 0.5
WITHIN = {"carbon": {"irrecoverable_carbon_m_soc": frac_soc, "irrecoverable_carbon_biomass": 1.0 - frac_soc}}

irrecoverable_carbon_biomass: cutoff 129.2 t/ha | tail 42 cells = 0.049% of PU (frozen T2 0.05%)
irrecoverable_carbon_m_soc: cutoff 267.8 t/ha | tail 4,384 cells = 5.150% of PU (frozen T2 5.15%)
            tail_cells  capture_mass  soc_claim_co_capture  independent
arm                                                                    
a0_control       42/42           1.0                   0.0          1.0
a1_protocol      42/42           1.0                   0.0          1.0
AB carbon mass split: SOC 67.5% / biomass 32.5% (parent 74.2/25.8)
a1 tail capture (mass) = 1.000 -> split rule: mass  [near-vacuous at AB tail size; disclosed]


## §B — S0–S5 by the parent §3.1 rules on the AB audit (rulings applied)

Four blocks at equal discretionary influence; biodiversity keeps **both** members (D-AB9);
carbon split per §A; S0–S3 targets = {m_soc: 0.322} (D2); S4 = carbon doubled + m_soc at the
ladder target (D3); S5 = S0 + gHM×10 (inexpressible push, mirror). Outside the accounting,
disclosed: gHM w = 1 and the 27 EFGs at 1/27. Weights derived on the **AB stack** at the parent
30% audit convention (`config.BUDGET_PCT`, C2 comparability); the 245 climate level is
re-derived at the AB-3 freeze exactly as the parent does.

In [13]:
# ---- derivation --------------------------------------------------------------------------------------
BASE_SHARES = {b: 1.0 / len(config.BLOCKS) for b in config.BLOCKS}
T_S0 = {"irrecoverable_carbon_m_soc": float(T2.loc["irrecoverable_carbon_m_soc", "target"])}
def doubled(block):
    n = len(config.BLOCKS)
    return {b: (2.0 / n if b == block else (1.0 - 2.0 / n) / (n - 1)) for b in config.BLOCKS}
SCENARIOS = {
    "S0_balanced":     (BASE_SHARES,             dict(T_S0)),
    "S1_core_habitat": (doubled("core_habitat"), dict(T_S0)),
    "S2_connectivity": (doubled("connectivity"), dict(T_S0)),
    "S3_biodiversity": (doubled("biodiversity"), dict(T_S0)),
    "S4_carbon":       (doubled("carbon"),       {"irrecoverable_carbon_m_soc": float(T_S4)}),
}
TBL = {name: lc.scenario_weights(sh, within_block=WITHIN, targets=tg, handoff_dir=AB)
       for name, (sh, tg) in SCENARIOS.items()}
# ssp245 climate level: constant intended influence, weights re-derived from the 245 realization's
# own swing (parent s3.2 mechanism; done here rather than at the freeze so AB-1 can solve both levels)
LP245 = {"climate_type_macrorefugia": AB / "climate_realizations" / "macrorefugia_245_2071_2100.tif"}
TBL245 = {name: lc.scenario_weights(sh, within_block=WITHIN, targets=tg, handoff_dir=AB, layer_paths=LP245)
          for name, (sh, tg) in SCENARIOS.items()}
t1 = pd.concat({k: v.set_index("feature")["w"] for k, v in TBL.items()}, axis=1)
t1.insert(0, "block", TBL["S0_balanced"].set_index("feature")["block"])
print("T1 skeleton -- derived AB weights (mean-1 per scenario); S5 = S0 with gHM x10 on top")
print(t1.round(3).to_string())
print("\ntargets: " + "; ".join(f"{k}: {tg}" for k, (_, tg) in SCENARIOS.items()))
wm = float(t1.loc["aoh_richness_mammals", "S0_balanced"])
print(f"\nD-AB9 disclosure: mammals derived w = {wm:.3f} in S0 (R3-inexpressible, leverage "
      f"{float(T2.loc['aoh_richness_mammals','leverage']):.3f}: capture confined to "
      f"{float(T2.loc['aoh_richness_mammals','cap_min']):.3f}-{float(T2.loc['aoh_richness_mammals','cap_max']):.3f} at ANY weight)")

# 245 realization on the AB stack: leverage + top-30% Jaccard vs 585 (parent Gate-1 SS C mirror)
tops = {}
for k in config.CLIMATE_REALIZATIONS:
    v = lc._read(AB / "climate_realizations" / f"macrorefugia_{k}.tif")[pu]
    cmin, cmax, lev = lc.leverage_of(v)
    tops[k] = v >= np.nanquantile(v, 1.0 - config.BUDGET_PCT)
    print(f"climate {k}: leverage {lev:.3f} [{cmin:.3f}, {cmax:.3f}]")
a, b = (tops[k] for k in config.CLIMATE_REALIZATIONS)
J_CLIM = float((a & b).sum() / (a | b).sum())
print(f"top-30% Jaccard between climate levels on AB = {J_CLIM:.3f} (parent 0.574)")

T1 skeleton -- derived AB weights (mean-1 per scenario); S5 = S0 with gHM x10 on top
                                     block  S0_balanced  S1_core_habitat  S2_connectivity  S3_biodiversity  S4_carbon
feature                                                                                                              
climate_type_macrorefugia     core_habitat        1.038            2.402            0.672            0.507      0.934
transboundary_connectivity    connectivity        0.306            0.236            0.595            0.150      0.276
climate_corridors             connectivity        1.598            1.232            3.105            0.780      1.438
irrecoverable_carbon_m_soc          carbon        0.219            0.169            0.142            0.107      0.587
irrecoverable_carbon_biomass        carbon        0.172            0.133            0.111            0.084      0.464
aoh_richness_birds            biodiversity        1.386            1.069            0.898

In [14]:
# ---- freeze: scenarios_ab_v1.json + ab_budget_levels_v1.json ------------------------------------------
now = datetime.now(timezone.utc).isoformat()
payload = {"_meta": dict(
    derived_utc=now, extent_id=EXTENT["extent_id"], mirror_spec_version="v0.14.1",
    audit_created_utc=CONSTS["created_utc"], n_pu=CONSTS["n_pu"],
    budget_pct_audit=config.BUDGET_PCT, budget_pct_solve_level_A=EXTENT["budget_pct_effective"],
    split_rule=SPLIT, frac_soc=round(frac_soc, 6), diagnostic_a1_tail_capture_mass=round(CAP, 6),
    biodiversity_block="both AOH members (D-AB9; mammals R3-inexpressible, weight inert, disclosed)",
    s4_ladder=dict(rungs={f"theta_{th}": t for th, t in LADDER}, measured_floor_a1_zero_pull=round(FLOOR_A, 6), a0_equal_weight_capture=round(a0_ms, 6),
                   proxy_floor=PROXY, margin=MARGIN, chosen_theta=TH_S4, chosen_target=T_S4),
    climate_top30_jaccard=round(J_CLIM, 4), blocks=config.BLOCKS, n_efg=len(lc.efg_paths(AB)),
    normalization="mean-1 over blocked features; outside fixed (gHM w=1, EFG 1/27)",
    layer_sha256=CONSTS["layer_sha256"])}
for name, (sh, tg) in SCENARIOS.items():
    d = TBL[name]
    payload[name] = dict(block_shares={k: round(v, 6) for k, v in sh.items()},
                         within_block={b: {f: round(x, 6) for f, x in m.items()} for b, m in WITHIN.items()},
                         targets=tg, weights={r.feature: round(r.w, 6) for r in d.itertuples()},
                         weights_ssp245={r.feature: round(r.w, 6) for r in TBL245[name].itertuples()},
                         intended_shares={r.feature: round(r.intended_share, 6) for r in d.itertuples()})
(SPEC / "scenarios_ab_v1.json").write_text(json.dumps(payload, indent=2))

n_ab, n_lock, n_unl, X = EXTENT["n_pu"], EXTENT["n_locked"], EXTENT["n_unlocked"], EXTENT["x_of_unlocked"]
levels = {}
for lab, x in (("A", X), ("B", X / 2)):
    cells = int(round(n_lock + x * n_unl))
    levels[lab] = dict(x_of_unlocked=x, budget_cells=cells, budget_pct=cells / n_ab, additions_cells=cells - n_lock,
                       role="mirror (parent realized fill rate)" if lab == "A" else "realistic envelope (X/2)")
assert levels["A"]["budget_cells"] == EXTENT["budget_cells"]
(SPEC / "ab_budget_levels_v1.json").write_text(json.dumps(dict(
    _meta=dict(created_utc=now, rule="D-AB5 v2: S0 nesting test at AB-1/AB-2; N = |core_B & core_A| / |core_B| >= 0.80 => A primary, else B primary",
               nesting_threshold=0.80, extent_id=EXTENT["extent_id"]), levels=levels), indent=2))
print(f"wrote spec/scenarios_ab_v1.json ({len(SCENARIOS)} scenarios; split {SPLIT}; S4 t={T_S4} @ theta {TH_S4}x)")
print("wrote spec/ab_budget_levels_v1.json: " + " | ".join(
    f"{k}: {v['budget_cells']:,} cells ({100*v['budget_pct']:.1f}%), additions {v['additions_cells']:,}" for k, v in levels.items()))

wrote spec/scenarios_ab_v1.json (5 scenarios; split mass; S4 t=0.772 @ theta 2.0x)
wrote spec/ab_budget_levels_v1.json: A: 38,055 cells (44.7%), additions 10,083 | B: 33,014 cells (38.8%), additions 5,042


## → next

Enter the AB-0 numbers in `spec/results_log.md` (R3) in this session: captures vs targets, the
measured floor vs proxy, the ladder outcome, a4 equivalence, the reallocation table, the split
diagnostic, the T1 skeleton, the climate Jaccard. Then `06_ab1_anchors` (S0 anchors × both budget
levels × both climate levels + LP twins; the D-AB5 v2 nesting test needs the guarded sweeps from
`07_ab2_mga`).